# Stage 4 — Option F: computational analysis

**Branch: feat/computational-analysis-experiment**

How much relative-depth accuracy does V2 gain over V1 at matching model sizes,
what inference cost accompanies that gain, and how does the accuracy–cost
tradeoff change across V2 Small, Base, and Large?

All model execution happens here in Colab. Choose **Runtime → Change runtime type → T4 GPU**.
Execute cells individually. Stop after downloading the pilot ZIP and return it
for review. Do not use Run all. The benchmark cell is gated separately.
Inherited accuracy and legacy CPU measurements will not be rerun.

## 1. Set up a fresh checkout and dependencies

In [ ]:
from pathlib import Path
import datetime, os, shutil, subprocess, sys, tempfile, zipfile
from google.colab import files

BRANCH = "feat/computational-analysis-experiment"
workspace = Path(tempfile.mkdtemp(prefix="dav2_stage4_", dir="/content"))
ROOT = workspace / "repo"
subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch",
                "https://github.com/ayyansohail/depth-anything-v2-reproduction.git", str(ROOT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                str(ROOT / "experiment/requirements-colab.txt")], check=True)
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR = ROOT / "experiment/runs" / ("stage4_" + stamp)
CHECK_DIR = ROOT / "experiment/runs" / ("checks_" + stamp)
CHECK_DIR.mkdir(parents=True)
print("Branch:", BRANCH)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip())
print("Run:", RUN_DIR)

def run_logged(arguments, log_path):
    env = dict(os.environ)
    env["PYTHONPATH"] = str(ROOT) + os.pathsep + str(ROOT / "src")
    env["PYTHONUNBUFFERED"] = "1"
    with Path(log_path).open("w", encoding="utf-8") as log:
        process = subprocess.Popen([sys.executable, *arguments], cwd=ROOT, env=env,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            result = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if result:
        raise RuntimeError(f"Command failed ({result}). Download evidence and return {log_path.name}.")

PILOT_REVIEWED = False

## 2. Validate source and saved evidence

These are lightweight Colab tests, not inference. A failure stops progress.
The code and notebook are also parsed for syntax here because no Python checks
were run locally during preparation.

In [ ]:
import ast, json
for folder in ("experiment", "tests"):
    for path in (ROOT / folder).rglob("*.py"):
        ast.parse(path.read_text(encoding="utf-8-sig"), filename=str(path))
notebook = json.loads((ROOT / "notebooks/stage4_computational_analysis.ipynb").read_text(encoding="utf-8-sig"))
for index, cell in enumerate(notebook["cells"]):
    if cell["cell_type"] == "code":
        ast.parse("".join(cell["source"]), filename=f"notebook_cell_{index}")
run_logged(["-m", "pytest", "-q", "tests/test_metrics.py", "tests/test_stage4_analysis.py"],
           CHECK_DIR / "validation.log")
print("Source parsing and tests passed.")

## 3. Capture this new environment and resolve model revisions

This checks for a T4, records the source commit, dependency snapshot and inherited
input checksums, and resolves immutable Hugging Face model revisions. It does not
recover the old accuracy environment. Run this once per new Colab runtime.

In [ ]:
assert "ROOT" in globals(), "Run setup in this session first."
run_logged(["experiment/benchmark_stage4.py", "--phase", "capture", "--run-dir", str(RUN_DIR)],
           CHECK_DIR / "capture.log")
shutil.copyfile(CHECK_DIR / "capture.log", RUN_DIR / "capture_console.log")

## 4. Run the six-model pilot

Weights download on first use. This verifies FP32 inputs/outputs and finite
predictions; it does not evaluate accuracy. If this fails, skip to the download
cell, return the ZIP, and do not start the benchmark.

In [ ]:
assert "RUN_DIR" in globals(), "Run setup, validation and capture first."
run_logged(["experiment/benchmark_stage4.py", "--phase", "pilot", "--run-dir", str(RUN_DIR)],
           RUN_DIR / "pilot_console.log")

## 5. Download pilot evidence and stop for review

Attach the ZIP in the chat. The helper also works after a failure and is reused
for the final download. It archives only evidence, not the model cache or weights.

In [ ]:
def download_evidence(label):
    archive = workspace / f"{RUN_DIR.name}_{label}.zip"
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
        if RUN_DIR.exists():
            for path in sorted(RUN_DIR.rglob("*")):
                if path.is_file():
                    z.write(path, str(path.relative_to(RUN_DIR)))
        for path in sorted(CHECK_DIR.rglob("*")):
            if path.is_file():
                z.write(path, "preflight/" + str(path.relative_to(CHECK_DIR)))
    print("Evidence:", archive)
    files.download(str(archive))

download_evidence("pilot")
print("STOP: return this ZIP for review before running the remaining cells.")

## 6. Benchmark after pilot review

Only change PILOT_REVIEWED to True after the pilot has been reviewed. Four sweeps
(A/B/B/A), six models each, ten warm-ups and fifty timed calls per block. Every
completed block is saved. No CPU, full-predict, training or accuracy rerun occurs.
If the runtime disconnects, do not splice another runtime into this run directory.

In [ ]:
PILOT_REVIEWED = False  # change to True only after returning the pilot ZIP for review
assert PILOT_REVIEWED, "Return the pilot ZIP first. Leave the benchmark paused until review."
run_logged(["experiment/benchmark_stage4.py", "--phase", "run", "--run-dir", str(RUN_DIR)],
           RUN_DIR / "benchmark_console.log")

## 7. Validate and analyze complete measurements

No inference in this cell. Incomplete or mixed evidence is rejected. Analysis
resamples paired image clusters, summarizes latency by sweep, and keeps inherited
accuracy and legacy CPU contexts explicit.

In [ ]:
run_logged(["experiment/analyze_stage4.py", "--run-dir", str(RUN_DIR)],
           RUN_DIR / "analysis_console.log")
from IPython.display import Image, display
for name in ("da2k_accuracy_vs_gpu_latency.png", "nyu_accuracy_vs_gpu_latency.png",
             "nyu_accuracy_vs_legacy_cpu.png", "order_sensitivity.png"):
    display(Image(filename=str(RUN_DIR / "analysis" / name)))

## 8. Download final evidence (or failure logs)

Return the ZIP in the chat. The assistant will check and import it on the
experiment branch before publishing findings. Do not claim Stage 4 complete
until that review is finished. Disconnecting Colab may delete all local files.

In [ ]:
download_evidence("final")